# Week 3: source-strength weaknesses and group training
The fixed full-data CNN analysis uses RFI-only test groups and its saved clean-validation FPR cutoff. It reproduces the midterm baseline values.

The later balanced-band experiment comes from `RFI_SNR_Band_Experiment_Concise.ipynb`: 18 RFI training recordings per band plus controls, with validation-F1 cutoffs. Its results describe that separate protocol. Reference source/noise ratios come from simulator metadata, before leakage and antenna gains.


In [ ]:
from IPython.display import display
from cs163_rfi import paths
SEED = 163
LR = 3e-4
BASE = 8
BINS = 65536

MIN_EPOCHS = 20
PATIENCE = 10
MIN_AP_IMPROVEMENT = 0.001
MAX_EPOCHS = 100

RFI_TRAIN_PER_BAND = 18
BASELINE_METHODS = ("sk", "bandpass", "sk-bandpass")
BASELINE_FAR = 1e-4

## Fixed full-data model
G1 is ratio below 0.1; G2 is 0.1 to below 1; G3+G4 is at least 1. Exclude ratio-zero clean/sky controls from these grouped metrics. The model and cutoff are frozen before test evaluation.

In [ ]:
import pandas as pd
RUN_FULL_MODEL = False  # Set True with the original full-data CNN checkpoint.
if RUN_FULL_MODEL:
    import json, os, torch
    from pathlib import Path
    from cs163_rfi.experiment import (
        SegmentationModel, inspect_dataset, sha256_file, evaluate_split,
        pooled_metrics, precision_recall_from_histograms,
    )
    calibration = json.loads((paths.RESULTS_DIR / "reference/source_strength_full_calibration.json").read_text())
    checkpoint_path = Path(os.environ.get("RFI_SNR_FULL_CHECKPOINT", str(paths.CHECKPOINT_DIR /
        "runs/grid_20261020T045116Z_selina_cnn_b8_18f27e4b/bce_lr0.0003/best.pt")))
    assert sha256_file(checkpoint_path) == calibration["best_checkpoint_sha256"]
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    full_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    full_model = SegmentationModel(8, False).to(full_device)
    full_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    _, full_catalog, _ = inspect_dataset(paths.REPO_ROOT, paths.DATA_DIR, None)
    test_rfi = full_catalog[full_catalog.split.eq("test") & full_catalog.reference_power_ratio.gt(0)]
    band_frames = {
        "low": test_rfi[test_rfi.reference_power_ratio.lt(.1)],
        "medium": test_rfi[test_rfi.reference_power_ratio.ge(.1) & test_rfi.reference_power_ratio.lt(1)],
        "high": test_rfi[test_rfi.reference_power_ratio.ge(1)],
    }
    full_rows = []
    for band, frame in band_frames.items():
        counts, ph, nh, _ = evaluate_split(full_model, paths.DATA_DIR, frame,
            checkpoint["normalization"], full_device, calibration["probability_bins"], calibration["quantized_cut"])
        metrics = pooled_metrics(counts, "method").loc["model_matched_clean_fpr"]
        full_rows.append(dict(band=band, test_recordings=len(frame),
            test_ap=precision_recall_from_histograms(ph, nh)["ap"],
            test_precision=metrics.precision, test_recall=metrics.recall,
            test_fpr=metrics.false_positive_rate))
    full_results = pd.DataFrame(full_rows)
    full_output = paths.RESULTS_DIR / "week_03/source_strength"
    full_output.mkdir(parents=True, exist_ok=True)
    full_results.to_csv(full_output / "full_model_groups.csv", index=False)
else:
    full_results = pd.read_csv(paths.RESULTS_DIR / "reference/source_strength_full_model.csv")
display(full_results[["band", "test_recordings", "test_ap", "test_precision", "test_recall", "test_fpr"]])

## Balanced-band training
This experiment balances training groups at 18 RFI recordings each and includes controls in validation/test. Keep its validation-F1 results separate from the fixed-model clean-FPR analysis above.

In [ ]:
from cs163_rfi import paths, imbalance as shared
from cs163_rfi import load_recording, flag_rfi, score_mask
from cs163_rfi.experiment import checkpoint_dir
from cs163_rfi.imbalance import CNN, make_loader as loader, add_hist, ap_from_hist as pixel_ap
from cs163_rfi.imbalance import score_dataset as score_cnn, best_f1_cut
from cs163_rfi.imbalance import FullDataset as RFIDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, random
from torch import nn
from IPython.display import display
import os
from pathlib import Path
OUT = Path(os.environ.get("RFI_SNR_RESULTS_DIR", str(paths.RESULTS_DIR / "snr_band_experiment")))
DATA = paths.DATA_DIR
OUT.mkdir(parents=True, exist_ok=True)
assert (SEED, LR, BASE, BINS) == (shared.SEED, shared.LR, shared.BASE, shared.BINS)
catalog = shared.prepare()
def band(r):
    if r == 0: return "control"
    if r < 0.1: return "low"
    if r < 1.0: return "medium"
    return "high"

catalog["snr_band"] = catalog.reference_power_ratio.map(band)
catalog["snr_db"] = np.where(
    catalog.reference_power_ratio > 0,
    10*np.log10(catalog.reference_power_ratio.where(catalog.reference_power_ratio > 0)),
    np.nan
)

counts = pd.crosstab(catalog.split, catalog.snr_band)
display(counts)

assert counts.loc["train", ["low","medium","high","control"]].to_dict() == {
    "low":18, "medium":33, "high":39, "control":10
}
assert counts.loc["validation", ["low","medium","high","control"]].to_dict() == {
    "low":18, "medium":20, "high":16, "control":6
}
assert counts.loc["test", ["low","medium","high","control"]].to_dict() == {
    "low":10, "medium":15, "high":11, "control":4
}

In [ ]:
controls = {
    s: catalog.query("split==@s and snr_band=='control'").copy()
    for s in ["train","validation","test"]
}

groups = {}
for b in ["low","medium","high"]:
    train_rfi = catalog.query(
        "split=='train' and snr_band==@b"
    ).sample(n=RFI_TRAIN_PER_BAND, random_state=SEED)

    groups[b] = {
        "train": pd.concat([train_rfi, controls["train"]]).sort_values("sample_id"),
        "validation": pd.concat([
            catalog.query("split=='validation' and snr_band==@b"),
            controls["validation"]
        ]).sort_values("sample_id"),
        "test": pd.concat([
            catalog.query("split=='test' and snr_band==@b"),
            controls["test"]
        ]).sort_values("sample_id"),
    }

display(pd.DataFrame([
    {
        "band":b,
        "train_RFI":int((groups[b]["train"].snr_band!="control").sum()),
        "validation_RFI":int((groups[b]["validation"].snr_band!="control").sum()),
        "test_RFI":int((groups[b]["test"].snr_band!="control").sum())
    }
    for b in ["low","medium","high"]
]))

def hard_metrics(ph, nh, cut):
    m = shared.metrics_at_cut(ph, nh, cut)
    return m["precision"], m["recall"], m["f1"], m["fpr"]


## Train three balanced-band CNNs
This is group retraining, separate from descriptive family analysis. All antennas retain their recording’s split; normalization uses all five original clean training recordings.


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)


def set_seed():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

RUN_TRAIN = False
if RUN_TRAIN:
    for b in ["low","medium","high"]:
        run=OUT/b
        run.mkdir(parents=True,exist_ok=True)

        set_seed()
        model=CNN(BASE).to(device)
        opt=torch.optim.Adam(model.parameters(),lr=LR)
        criterion=nn.BCEWithLogitsLoss()

        train_set=RFIDataset(groups[b]["train"])
        val_set=RFIDataset(groups[b]["validation"])
        train_loader=loader(train_set,True,SEED)

        best_ap=-np.inf; anchor_ap=-np.inf
        best_epoch=0; anchor_epoch=0; hist=[]

        for epoch in range(1,MAX_EPOCHS+1):
            model.train(); loss_sum=0.0; pixels=0
            for x,y in train_loader:
                x,y=x.to(device),y.to(device)
                loss=criterion(model(x),y)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                loss_sum += loss.item()*y.numel(); pixels += y.numel()

            vph,vnh=score_cnn(model,val_set,device)
            vap=pixel_ap(vph,vnh)
            hist.append({"epoch":epoch,"train_loss":loss_sum/pixels,
                         "validation_ap":vap})

            if vap>best_ap:
                best_ap=vap; best_epoch=epoch
                torch.save(model.state_dict(),checkpoint_dir(run)/"best.pt")

            if vap>anchor_ap+MIN_AP_IMPROVEMENT:
                anchor_ap,anchor_epoch=vap,epoch

            pd.DataFrame(hist).to_csv(run/"history.csv",index=False)
            print(f"{b:6s} epoch {epoch:02d} | AP {vap:.5f} | best {best_ap:.5f}")

            if epoch>=MIN_EPOCHS and epoch-anchor_epoch>=PATIENCE:
                break

        if epoch-anchor_epoch < PATIENCE:
            raise RuntimeError("Epoch budget exhausted before plateau")
        torch.cuda.empty_cache()

## Freeze thresholds, then evaluate
Enable after reviewing the validation runs. Each group uses its own validation-F1 cutoff; the test set has already been examined historically.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    def baseline_metrics(frame, method):
        totals = dict(tp=0,fp=0,fn=0,tn=0)
    
        for row in frame.itertuples():
            rec = load_recording(DATA/row.path)
            mask = flag_rfi(rec["power"], method=method, far=BASELINE_FAR)
            s = score_mask(mask, rec["rfi_mask"])
            for k in totals:
                totals[k] += int(s[k])
    
        tp,fp,fn,tn = [totals[k] for k in ["tp","fp","fn","tn"]]
        precision = tp/(tp+fp) if tp+fp else np.nan
        recall = tp/(tp+fn) if tp+fn else np.nan
        fpr = fp/(fp+tn) if fp+tn else np.nan
        removed = (tp+fp)/(tp+fp+fn+tn)
        return precision,recall,fpr,removed
    
    rows=[]
    
    for b in ["low","medium","high"]:
        model=CNN(BASE).to(device)
        model.load_state_dict(torch.load(
            checkpoint_dir(OUT/b)/"best.pt",map_location=device,weights_only=True))
    
        val_ds=RFIDataset(groups[b]["validation"])
        test_ds=RFIDataset(groups[b]["test"])
    
        vph,vnh=score_cnn(model,val_ds,device)
        cut,val_f1=best_f1_cut(vph,vnh)
    
        tph,tnh=score_cnn(model,test_ds,device)
        p,r,f1,fpr=hard_metrics(tph,tnh,cut)
    
        rows.append({
            "band":b,"detector":"CNN",
            "validation_ap":pixel_ap(vph,vnh),
            "test_ap":pixel_ap(tph,tnh),
            "test_precision":p,"test_recall":r,
            "test_fpr":fpr
        })
    
        for method in BASELINE_METHODS:
            p,r,fpr,removed=baseline_metrics(groups[b]["test"],method)
            rows.append({
                "band":b,"detector":method,
                "validation_ap":np.nan,"test_ap":np.nan,
                "test_precision":p,"test_recall":r,
                "test_fpr":fpr
            })
    
    results=pd.DataFrame(rows)
    results.to_csv(OUT/"snr_band_detector_comparison.csv",index=False)
    display(results)
    
    # Main starter-style result: recall versus source-power band.
    pivot=results.pivot(index="band",columns="detector",values="test_recall")
    pivot=pivot.loc[["low","medium","high"]]
    
    plt.figure(figsize=(7,4))
    for detector in ["sk","bandpass","sk-bandpass","CNN"]:
        plt.plot(["< -10 dB","-10 to <0 dB",">= 0 dB"],
                 pivot[detector],marker="o",label=detector)
    plt.xlabel("Reference source/noise power ratio")
    plt.ylabel("Test recall")
    plt.title("Recall versus source strength")
    plt.legend()
    plt.grid(alpha=.2)
    plt.tight_layout()
    plt.savefig(OUT/"snr_band_recall_comparison.png",dpi=200)
    plt.show()
    
    print("Saved:",OUT)


In [ ]:
display(pd.read_csv(paths.RESULTS_DIR / "reference/snr_band_comparison.csv"))
print("Saved:", OUT)
